# Exhaustive vs Greedy 2-Patch Grid Defense — Test

**Purpose**: test whether replacing the greedy 2-patch search (15 candidates) with an
exhaustive search over all C(16,2) = 120 patch pairs meaningfully improves defense accuracy.

Runs on the **100-image tune subset** (10/class) using the **multilingual attack** to keep
runtime manageable.

Compares three methods:
- `grid_1patch` — best single patch (16 candidates)
- `grid_2patch_greedy` — fix best 1st patch, try 15 remaining (62 total fwd passes)
- `grid_2patch_exhaustive` — try all C(16,2)=120 pairs (240 total fwd passes)

Results saved to `results/comparison.json`.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'open_clip_torch', 'transformers', 'datasets', 'matplotlib', 'Pillow'], check=False)

In [ ]:
import os, random, json, time
from itertools import combinations
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import open_clip
from PIL import Image, ImageDraw, ImageFont
from datasets import load_dataset
from transformers import ChineseCLIPModel, ChineseCLIPProcessor

os.makedirs('results', exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

LANGS = ['en', 'zh']
CLASSES = {
    'en': ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck'],
    'zh': ['飞机', '汽车', '鸟', '猫', '鹿', '狗', '青蛙', '马', '船', '卡车'],
}
TMPL = {'en': 'a photo of a {}.', 'zh': '一张{}的照片。'}
DISPLAY_SIZE = 224
NUM_BOXES    = 2
FONT_SIZE    = 24
PAD          = 8

## 1. Models

In [ ]:
def _clip_feat(out):
    if torch.is_tensor(out): return out
    if getattr(out, 'pooler_output', None) is not None: return out.pooler_output
    raise TypeError(type(out))

class EnCLIP:
    lang = 'en'
    def __init__(self):
        self.m, _, self.pp = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
        self.m = self.m.to(DEVICE).eval()
        self.tok = open_clip.get_tokenizer('ViT-B-32')
    @torch.no_grad()
    def embed_images(self, imgs):
        x = torch.stack([self.pp(im) for im in imgs]).to(DEVICE)
        return F.normalize(self.m.encode_image(x), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.tok([TMPL['en'].format(w) for w in words]).to(DEVICE)
        return F.normalize(self.m.encode_text(t), dim=-1)

class ZhCLIP:
    lang = 'zh'
    def __init__(self):
        self.m = ChineseCLIPModel.from_pretrained('OFA-Sys/chinese-clip-vit-base-patch16').to(DEVICE).eval()
        self.p = ChineseCLIPProcessor.from_pretrained('OFA-Sys/chinese-clip-vit-base-patch16')
    @torch.no_grad()
    def embed_images(self, imgs):
        pv = self.p(images=imgs, return_tensors='pt').pixel_values.to(DEVICE)
        return F.normalize(_clip_feat(self.m.get_image_features(pixel_values=pv)), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.p(text=[TMPL['zh'].format(w) for w in words], padding=True, return_tensors='pt').to(DEVICE)
        out = self.m.get_text_features(input_ids=t['input_ids'], attention_mask=t['attention_mask'],
                                        token_type_ids=t.get('token_type_ids'))
        return F.normalize(_clip_feat(out), dim=-1)

def classify(model, imgs, words, batch_size=128):
    preds = []
    for i in range(0, len(imgs), batch_size):
        imf = model.embed_images(imgs[i:i+batch_size])
        tf  = model.embed_texts(words)
        preds.append((imf @ tf.t()).argmax(-1).cpu().numpy())
    return np.concatenate(preds)

def classify_conf(model, imgs, words, batch_size=128):
    preds, confs = [], []
    for i in range(0, len(imgs), batch_size):
        imf = model.embed_images(imgs[i:i+batch_size])
        tf  = model.embed_texts(words)
        sims = imf @ tf.t()
        preds.append(sims.argmax(-1).cpu().numpy())
        confs.append(sims.max(-1).values.cpu().numpy())
    return np.concatenate(preds), np.concatenate(confs)

print('Model classes defined.')

## 2. Dataset + attack

In [ ]:
hf = load_dataset('uoft-cs/cifar10', split='test')
label_key = 'label' if 'label' in hf.column_names else 'labels'
image_key = 'img'   if 'img'   in hf.column_names else 'image'

with open('../../image_samples/CIFAR10_BALANCED_1000_SAMPLE.json', encoding='utf-8') as f:
    _saved = json.load(f)

idx  = _saved['idx']
rows = hf.select(idx)
true = np.array(rows[label_key])

rng    = random.Random(0)
target = np.array([rng.choice([c for c in range(10) if c != int(true[k])]) for k in range(len(idx))])

clean_224 = [im.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC)
             for im in rows[image_key]]

# Tune subset: 10 per class = 100 images
tune_idx = np.concatenate([np.where(true == c)[0][:10] for c in range(10)])
print(f'Tune subset size: {len(tune_idx)}')

In [ ]:
# Multilingual attack: box-0=EN word, box-1=ZH word
import platform
_FONT_CACHE = {}

def _font_paths():
    if platform.system() == 'Windows':
        winfonts = os.path.join(os.environ.get('WINDIR', r'C:\Windows'), 'Fonts')
        cjk   = os.path.join(winfonts, 'msyh.ttc')
        latin = os.path.join(winfonts, 'arial.ttf')
        if not os.path.exists(latin): latin = cjk
        return (cjk if os.path.exists(cjk) else None, latin if os.path.exists(latin) else None)
    for d in ['/usr/share/fonts', '/Library/Fonts', os.path.expanduser('~/.fonts')]:
        for f in ['NotoSansCJK-Regular.ttc', 'NotoSans-Regular.ttf']:
            p = os.path.join(d, f)
            if os.path.exists(p): return p, p
    return None, None

_CJK_FONT, _LAT_FONT = _font_paths()

def _get_font(fp):
    if fp not in _FONT_CACHE:
        try:    _FONT_CACHE[fp] = ImageFont.truetype(fp, FONT_SIZE) if fp else ImageFont.load_default()
        except: _FONT_CACHE[fp] = ImageFont.load_default()
    return _FONT_CACHE[fp]

def _rects_overlap(a, b):
    return not (a[2] <= b[0] or b[2] <= a[0] or a[3] <= b[1] or b[3] <= a[1])

def _random_nonoverlapping_rect(rng, bw, bh, placed):
    x_hi = max(0, DISPLAY_SIZE - bw); y_hi = max(0, DISPLAY_SIZE - bh)
    rx, ry = 0, 0
    for _ in range(64):
        rx = rng.randint(0, x_hi) if x_hi > 0 else 0
        ry = rng.randint(0, y_hi) if y_hi > 0 else 0
        rect = (rx, ry, rx+bw, ry+bh)
        if all(not _rects_overlap(rect, p) for p in placed): return rect
    return (rx, ry, rx+bw, ry+bh)

def draw_multilingual_attack(img, en_word, zh_word, img_idx):
    img = img.copy()
    draw = ImageDraw.Draw(img)
    placed = []
    for box_i, (word, fp) in enumerate([(en_word, _LAT_FONT), (zh_word, _CJK_FONT)]):
        font = _get_font(fp)
        bb   = draw.textbbox((0, 0), word, font=font)
        bw   = (bb[2]-bb[0]) + 2*PAD
        bh   = (bb[3]-bb[1]) + PAD + 12
        rng_box = random.Random(int(img_idx) * NUM_BOXES + box_i)
        rect = _random_nonoverlapping_rect(rng_box, bw, bh, placed)
        placed.append(rect)
        rx, ry, rx2, ry2 = rect
        draw.rectangle([rx, ry, rx2, ry2], fill='white')
        draw.text((rx+PAD-bb[0], ry+PAD-bb[1]), word, fill='black', font=font)
    return img

print('Building multilingual attacked images for tune subset...')
attacked = [draw_multilingual_attack(clean_224[i],
                                      CLASSES['en'][target[i]],
                                      CLASSES['zh'][target[i]], i)
            for i in tune_idx]
print(f'Built {len(attacked)} attacked images.')

## 3. Load models + baseline

In [ ]:
models = {}
for lang, cls in [('en', EnCLIP), ('zh', ZhCLIP)]:
    t0 = time.time()
    print(f'Loading {lang}...', end=' ', flush=True)
    models[lang] = cls()
    print(f'{time.time()-t0:.1f}s')

true_tune   = true[tune_idx]
target_tune = target[tune_idx]

preds_atk = {ml: classify(models[ml], attacked, CLASSES[ml]) for ml in LANGS}
print('Attacked acc:',  {ml: f'{100*(preds_atk[ml]==true_tune).mean():.1f}%' for ml in LANGS})
print('Attacked ASR:',  {ml: f'{100*(preds_atk[ml]==target_tune).mean():.1f}%' for ml in LANGS})

## 4. Grid helpers

In [ ]:
GRID_ROWS = GRID_COLS = 4
PATCH_H = PATCH_W = DISPLAY_SIZE // GRID_ROWS  # 56 px
PATCHES = [
    (c*PATCH_W, r*PATCH_H, (c+1)*PATCH_W, (r+1)*PATCH_H)
    for r in range(GRID_ROWS) for c in range(GRID_COLS)
]
ALL_PAIRS = list(combinations(range(len(PATCHES)), 2))  # C(16,2) = 120
print(f'{len(PATCHES)} patches, {len(ALL_PAIRS)} exhaustive pairs')

def occlude_rect(arr, rect, fill):
    out = arr.copy()
    x0, y0, x1, y1 = rect
    out[y0:y1, x0:x1] = fill
    return out

def score_candidates(candidates):
    """Mean max-cosine-sim across EN and ZH models — higher = more model-confident."""
    scores = np.zeros(len(candidates))
    for ml in LANGS:
        _, confs = classify_conf(models[ml], candidates, CLASSES[ml])
        scores += confs
    return scores / len(LANGS)

## 5. Run all three methods

In [ ]:
def run_1patch(imgs):
    result, best = [], []
    for j, img in enumerate(imgs):
        arr  = np.array(img.convert('RGB'))
        fill = arr.reshape(-1, 3).mean(0).astype(np.uint8)
        cands = [Image.fromarray(occlude_rect(arr, r, fill)) for r in PATCHES]
        scores = score_candidates(cands)
        bi = int(scores.argmax())
        best.append(bi); result.append(cands[bi])
    return result, best

def run_2patch_greedy(imgs, first_patches):
    result, seconds = [], []
    for j, (img, fp) in enumerate(zip(imgs, first_patches)):
        arr  = np.array(img.convert('RGB'))
        fill = arr.reshape(-1, 3).mean(0).astype(np.uint8)
        arr1 = occlude_rect(arr, PATCHES[fp], fill)
        remain = [i for i in range(len(PATCHES)) if i != fp]
        cands  = [Image.fromarray(occlude_rect(arr1, PATCHES[pi], fill)) for pi in remain]
        scores = score_candidates(cands)
        bi = int(scores.argmax())
        seconds.append(remain[bi]); result.append(cands[bi])
    return result, seconds

def run_2patch_exhaustive(imgs):
    """Try all C(16,2)=120 pairs, keep the best-scoring pair per image."""
    result, best_pairs = [], []
    for j, img in enumerate(imgs):
        arr  = np.array(img.convert('RGB'))
        fill = arr.reshape(-1, 3).mean(0).astype(np.uint8)
        cands = []
        for p1, p2 in ALL_PAIRS:
            a = occlude_rect(arr,  PATCHES[p1], fill)
            a = occlude_rect(a,    PATCHES[p2], fill)
            cands.append(Image.fromarray(a))
        scores = score_candidates(cands)
        bi = int(scores.argmax())
        best_pairs.append(ALL_PAIRS[bi]); result.append(cands[bi])
        if (j+1) % 20 == 0:
            print(f'  exhaustive {j+1}/{len(imgs)}')
    return result, best_pairs

print('Helpers ready.')

In [ ]:
print('--- 1-patch ---')
t0 = time.time()
res_1p, best_1p = run_1patch(attacked)
t_1p = time.time() - t0
print(f'Done in {t_1p:.1f}s')

print('\n--- 2-patch greedy ---')
t0 = time.time()
res_greedy, best_greedy_2 = run_2patch_greedy(attacked, best_1p)
t_greedy = time.time() - t0
print(f'Done in {t_greedy:.1f}s')

print('\n--- 2-patch exhaustive ---')
t0 = time.time()
res_exh, best_exh = run_2patch_exhaustive(attacked)
t_exh = time.time() - t0
print(f'Done in {t_exh:.1f}s')

## 6. Evaluate and compare

In [ ]:
def eval_imgs(imgs, label):
    row = {'method': label}
    for ml in LANGS:
        p = classify(models[ml], imgs, CLASSES[ml])
        row[f'acc_{ml}']  = float((p == true_tune).mean())
        row[f'asr_{ml}']  = float((p == target_tune).mean())
    row['acc_mean'] = float(np.mean([row[f'acc_{ml}'] for ml in LANGS]))
    row['asr_mean'] = float(np.mean([row[f'asr_{ml}'] for ml in LANGS]))
    return row

baseline_row = eval_imgs(attacked, 'no_defense')
row_1p       = eval_imgs(res_1p,   'grid_1patch')
row_greedy   = eval_imgs(res_greedy, 'grid_2patch_greedy')
row_exh      = eval_imgs(res_exh,    'grid_2patch_exhaustive')

rows = [baseline_row, row_1p, row_greedy, row_exh]

print(f'\n{"Method":<26} {"EN acc":>8} {"ZH acc":>8} {"Mean acc":>10} {"EN ASR":>8} {"ZH ASR":>8}')
print('-' * 72)
for r in rows:
    print(f'{r["method"]:<26} {100*r["acc_en"]:>7.1f}% {100*r["acc_zh"]:>7.1f}% '
          f'{100*r["acc_mean"]:>9.1f}% {100*r["asr_en"]:>7.1f}% {100*r["asr_zh"]:>7.1f}%')

print(f'\nRuntime — 1patch: {t_1p:.1f}s  greedy: {t_greedy:.1f}s  exhaustive: {t_exh:.1f}s')
print(f'Exhaustive gain over greedy: {100*(row_exh["acc_mean"]-row_greedy["acc_mean"]):+.1f}pp mean acc')

## 7. How often does greedy pick a suboptimal pair?

In [ ]:
# For each image, check whether greedy picked the same pair as exhaustive
greedy_pairs = list(zip(best_1p, best_greedy_2))
greedy_pairs_sorted = [tuple(sorted(p)) for p in greedy_pairs]
exh_pairs_sorted    = [tuple(sorted(p)) for p in best_exh]

n_agree = sum(g == e for g, e in zip(greedy_pairs_sorted, exh_pairs_sorted))
n_total = len(greedy_pairs_sorted)
print(f'Greedy and exhaustive agree on same pair: {n_agree}/{n_total} images ({100*n_agree/n_total:.1f}%)')
print(f'Greedy suboptimal on: {n_total-n_agree}/{n_total} images ({100*(n_total-n_agree)/n_total:.1f}%)')

## 8. Plot — accuracy comparison

In [ ]:
methods = [r['method'] for r in rows]
acc_en  = [100*r['acc_en']   for r in rows]
acc_zh  = [100*r['acc_zh']   for r in rows]
acc_mn  = [100*r['acc_mean'] for r in rows]

x = np.arange(len(methods))
w = 0.25
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w, acc_en, w, label='EN acc', color='steelblue')
ax.bar(x,     acc_zh, w, label='ZH acc', color='tomato')
ax.bar(x + w, acc_mn, w, label='Mean acc', color='seagreen')
ax.set_xticks(x)
ax.set_xticklabels([m.replace('_', '\n') for m in methods], fontsize=9)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Grid defense: greedy vs exhaustive 2-patch\n(100-image tune subset, multilingual attack)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('results/greedy_vs_exhaustive.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved -> results/greedy_vs_exhaustive.png')

## 9. Save results

In [ ]:
output = {
    'description': 'Greedy vs exhaustive 2-patch grid defense on 100-image tune subset (multilingual attack)',
    'n_images': len(tune_idx),
    'n_patch_pairs_exhaustive': len(ALL_PAIRS),
    'runtime_seconds': {'grid_1patch': t_1p, 'grid_2patch_greedy': t_greedy, 'grid_2patch_exhaustive': t_exh},
    'results': rows,
    'greedy_suboptimal_count': n_total - n_agree,
    'greedy_suboptimal_pct': round(100*(n_total-n_agree)/n_total, 1),
}
with open('results/comparison.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print('Saved -> results/comparison.json')